# Computational Notebook 15: Multichain Analysis

## Overview

The blockchain ecosystem has evolved from a single-chain world (Bitcoin) to a multichain landscape with dozens of Layer 1 (L1) blockchains and Layer 2 (L2) scaling solutions competing for users, developers, and capital. This notebook quantifies the tradeoffs between chains using the blockchain trilemma framework, compares L1 performance metrics, analyzes L2 scaling approaches, models cross-chain bridge security, and builds dashboards for tracking capital flows across the ecosystem.

## Prerequisites
- **Notebook 03**: Ethereum & EVM Analysis (EVM architecture, gas model)
- **Notebook 06**: Market Analysis (price data, market metrics)
- Basic Python programming and familiarity with NumPy

## Learning Objectives

1. Quantify the blockchain trilemma tradeoffs across security, scalability, and decentralization
2. Compare Layer 1 blockchains across TPS, finality, fees, and validator counts
3. Analyze Layer 2 scaling solutions: optimistic rollups, ZK rollups, state channels, and sidechains
4. Model cross-chain bridge architectures and quantify their security risks
5. Simulate interoperability protocols and atomic swaps
6. Build chain metrics dashboards with synthetic but realistic data
7. Analyze capital migration patterns between chains

**Estimated Time:** 4-6 hours

**Related Content:** [Section 05: Platform Comparison](../sections/05-platform-comparison.md)

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field
from collections import defaultdict

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True

print("All imports successful!")
print("This notebook analyzes multichain blockchain ecosystems.")

---
## 1. The Blockchain Trilemma

Vitalik Buterin articulated the blockchain trilemma: a blockchain can optimize for at most two of three properties simultaneously:

1. **Security** -- Resistance to attacks, immutability of history
2. **Scalability** -- Ability to process many transactions quickly
3. **Decentralization** -- Distribution of control, permissionlessness

> **Definition: Blockchain Trilemma** -- The observation that blockchain systems face fundamental tradeoffs between security, scalability, and decentralization. Improving one dimension typically requires sacrificing another.

We can quantify each dimension:
- **Security**: Cost to attack (51% attack cost), validator stake, audit quality
- **Scalability**: Transactions per second (TPS), time to finality, gas fees
- **Decentralization**: Nakamoto coefficient, number of validators, geographic distribution

**Source:** Buterin, V. (2021). "Why sharding is great." vitalik.eth.limo

In [ ]:
@dataclass
class ChainProfile:
    """Performance profile of a blockchain."""
    name: str
    consensus: str
    tps: float                    # Transactions per second
    finality_seconds: float       # Time to finality
    avg_fee_usd: float            # Average transaction fee
    validators: int               # Number of validators/miners
    nakamoto_coeff: int           # Nakamoto coefficient
    attack_cost_b: float          # Estimated 51% attack cost ($B)
    tvl_b: float                  # Total Value Locked ($B)
    daily_txns_m: float           # Daily transactions (millions)
    launch_year: int
    category: str                 # L1, L2, sidechain


# Synthetic but realistic chain data
chains = [
    ChainProfile("Bitcoin", "PoW", 7, 3600, 5.0, 15000, 4, 20.0, 0, 0.3, 2009, "L1"),
    ChainProfile("Ethereum", "PoS", 30, 768, 3.0, 900000, 6, 35.0, 50.0, 1.2, 2015, "L1"),
    ChainProfile("Solana", "PoH+PoS", 4000, 0.4, 0.001, 2000, 19, 1.5, 4.0, 30.0, 2020, "L1"),
    ChainProfile("Avalanche", "Snow", 4500, 2, 0.05, 1500, 26, 0.8, 1.5, 2.0, 2020, "L1"),
    ChainProfile("Cardano", "Ouroboros", 250, 600, 0.20, 3000, 25, 0.5, 0.3, 0.1, 2017, "L1"),
    ChainProfile("BNB Chain", "PoSA", 2000, 3, 0.10, 29, 7, 0.3, 5.0, 4.0, 2020, "L1"),
    ChainProfile("Polygon PoS", "PoS", 7000, 128, 0.01, 100, 4, 0.1, 1.0, 3.0, 2020, "L1/Sidechain"),
    ChainProfile("Arbitrum", "Optimistic", 4000, 604800, 0.10, 1, 1, 0, 3.0, 2.0, 2021, "L2"),
    ChainProfile("Optimism", "Optimistic", 2000, 604800, 0.10, 1, 1, 0, 1.5, 1.0, 2021, "L2"),
    ChainProfile("zkSync Era", "ZK Rollup", 2000, 3600, 0.15, 1, 1, 0, 0.5, 0.8, 2023, "L2"),
]

print("=" * 90)
print("LAYER 1 & LAYER 2 BLOCKCHAIN COMPARISON")
print("=" * 90)

print(f"\n{'Chain':<15} {'Type':<12} {'TPS':>6} {'Finality':>10} {'Fee ($)':>8} {'Validators':>11} {'Nakamoto':>9}")
print("-" * 75)
for c in chains:
    if c.finality_seconds >= 86400:
        fin = f"{c.finality_seconds/86400:.0f}d"
    elif c.finality_seconds >= 60:
        fin = f"{c.finality_seconds/60:.0f}m"
    else:
        fin = f"{c.finality_seconds:.1f}s"
    print(f"{c.name:<15} {c.category:<12} {c.tps:>6,.0f} {fin:>10} ${c.avg_fee_usd:>6.3f} "
          f"{c.validators:>11,} {c.nakamoto_coeff:>9}")

In [ ]:
# Trilemma radar charts
def trilemma_scores(chain: ChainProfile) -> Tuple[float, float, float]:
    """Calculate trilemma scores (0-10) for security, scalability, decentralization."""
    # Security: based on attack cost and validator count
    security = min(10, np.log10(chain.attack_cost_b * 1e9 + 1) * 1.1 +
                  min(3, np.log10(chain.validators + 1)))
    
    # Scalability: TPS and fee efficiency
    scalability = min(10, np.log10(chain.tps + 1) * 2.5 +
                     max(0, 3 - np.log10(chain.avg_fee_usd + 0.001)))
    
    # Decentralization: Nakamoto coefficient and validator count
    decentralization = min(10, chain.nakamoto_coeff * 0.3 +
                          np.log10(chain.validators + 1) * 1.2)
    
    return security, scalability, decentralization


# Calculate scores
l1_chains = [c for c in chains if c.category == "L1"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw=dict(polar=True))
axes = axes.flatten()

categories = ['Security', 'Scalability', 'Decentralization']
angles = np.linspace(0, 2 * np.pi, 3, endpoint=False).tolist()
angles += angles[:1]

colors = ['orange', 'blue', 'purple', 'red', 'green', 'brown']

for ax, chain, color in zip(axes, l1_chains, colors):
    scores = list(trilemma_scores(chain))
    scores += scores[:1]
    
    ax.plot(angles, scores, 'o-', linewidth=2, color=color)
    ax.fill(angles, scores, alpha=0.25, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, fontsize=8)
    ax.set_ylim(0, 10)
    ax.set_title(chain.name, fontsize=12, fontweight='bold')

plt.suptitle('Blockchain Trilemma: Security vs Scalability vs Decentralization',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/trilemma_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Each chain makes different tradeoffs in the trilemma.")

---
## 2. Layer 1 Performance Comparison

Layer 1 blockchains differ fundamentally in their architecture, which determines their performance characteristics.

In [ ]:
# Comprehensive L1 comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

l1_names = [c.name for c in l1_chains]
l1_colors = ['orange', 'blue', 'purple', 'red', 'green', 'brown']

# TPS comparison
tps_vals = [c.tps for c in l1_chains]
bars = axes[0, 0].barh(l1_names, tps_vals, color=l1_colors)
axes[0, 0].set_xlabel('TPS')
axes[0, 0].set_title('Throughput (Transactions Per Second)')
axes[0, 0].set_xscale('log')
for bar, tps in zip(bars, tps_vals):
    axes[0, 0].text(bar.get_width() * 1.2, bar.get_y() + bar.get_height()/2,
                    f'{tps:,.0f}', va='center', fontsize=9)

# Fee comparison
fees = [c.avg_fee_usd for c in l1_chains]
bars = axes[0, 1].barh(l1_names, fees, color=l1_colors)
axes[0, 1].set_xlabel('Average Fee ($)')
axes[0, 1].set_title('Transaction Fees')
axes[0, 1].set_xscale('log')

# Validator count
validators = [c.validators for c in l1_chains]
bars = axes[1, 0].barh(l1_names, validators, color=l1_colors)
axes[1, 0].set_xlabel('Validators')
axes[1, 0].set_title('Validator Count')
axes[1, 0].set_xscale('log')

# Scatter: TPS vs Validators (scalability vs decentralization)
for c, color in zip(l1_chains, l1_colors):
    axes[1, 1].scatter(c.validators, c.tps, s=c.tvl_b * 30 + 50,
                       color=color, label=c.name, zorder=5, alpha=0.7)
    axes[1, 1].annotate(c.name, (c.validators, c.tps), fontsize=8,
                        xytext=(5, 5), textcoords='offset points')

axes[1, 1].set_xlabel('Validators (log scale)')
axes[1, 1].set_ylabel('TPS (log scale)')
axes[1, 1].set_title('Scalability vs Decentralization (size = TVL)')
axes[1, 1].set_xscale('log')
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.savefig('/tmp/l1_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Clear tradeoff: higher TPS chains tend to have fewer validators.")

---
## 3. Layer 2 Scaling Solutions

Layer 2 solutions process transactions off the main chain while inheriting its security:

> **Definition: Optimistic Rollup** -- An L2 that posts transaction data to L1 and assumes transactions are valid by default. Invalid transactions can be challenged during a dispute period (typically 7 days) via fraud proofs.

> **Definition: ZK (Zero-Knowledge) Rollup** -- An L2 that posts transaction data along with a cryptographic validity proof (ZK-SNARK/STARK) to L1. No challenge period needed since validity is mathematically proven.

| Type | Examples | Proof | Withdrawal Time | Data On-Chain |
|------|----------|-------|-----------------|---------------|
| Optimistic Rollup | Arbitrum, Optimism | Fraud proof | ~7 days | Yes |
| ZK Rollup | zkSync, StarkNet | Validity proof | Minutes | Yes |
| State Channel | Lightning Network | Signature exchange | Instant | No |
| Sidechain | Polygon PoS | Independent consensus | Minutes | No |

**Source:** Thibault, L. et al. (2022). "Blockchain Scaling Using Rollups." *Algorithms*.

In [ ]:
@dataclass
class L2Profile:
    """Layer 2 scaling solution profile."""
    name: str
    type: str
    tps: float
    withdrawal_time: str
    data_on_l1: bool
    security_from_l1: bool
    compression_ratio: float   # TX data compression vs L1
    proof_cost_gas: int        # Gas cost to verify proof on L1
    tvl_b: float


l2_solutions = [
    L2Profile("Arbitrum One", "Optimistic", 4000, "7 days", True, True, 10, 0, 3.0),
    L2Profile("Optimism", "Optimistic", 2000, "7 days", True, True, 10, 0, 1.5),
    L2Profile("zkSync Era", "ZK Rollup", 2000, "~1 hour", True, True, 100, 500000, 0.5),
    L2Profile("StarkNet", "ZK Rollup", 1000, "~1 hour", True, True, 200, 300000, 0.3),
    L2Profile("Lightning", "State Channel", 100000, "Instant", False, False, 1000, 0.2),
    L2Profile("Polygon PoS", "Sidechain", 7000, "~30 min", False, False, 1, 0, 1.0),
]

def l2_throughput_multiplier(l2: L2Profile, l1_tps: float = 30) -> float:
    """Calculate throughput multiplier over L1."""
    return l2.tps / l1_tps


print("=" * 80)
print("LAYER 2 SCALING COMPARISON")
print("=" * 80)

print(f"\n{'Solution':<16} {'Type':<12} {'TPS':>6} {'Withdrawal':>12} {'L1 Security':>12} {'Multiplier':>11}")
print("-" * 72)
for l2 in l2_solutions:
    mult = l2_throughput_multiplier(l2)
    sec = 'Yes' if l2.security_from_l1 else 'No'
    print(f"{l2.name:<16} {l2.type:<12} {l2.tps:>6,.0f} {l2.withdrawal_time:>12} {sec:>12} {mult:>10.0f}x")

# Visualize L2 tradeoffs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TPS comparison
l2_names = [l.name for l in l2_solutions]
l2_tps = [l.tps for l in l2_solutions]
type_colors = {'Optimistic': 'blue', 'ZK Rollup': 'green',
               'State Channel': 'purple', 'Sidechain': 'orange'}
l2_colors = [type_colors[l.type] for l in l2_solutions]

bars = axes[0].barh(l2_names, l2_tps, color=l2_colors)
axes[0].axvline(x=30, color='red', linestyle='--', alpha=0.5, label='Ethereum L1 (30 TPS)')
axes[0].set_xlabel('TPS')
axes[0].set_title('L2 Throughput')
axes[0].set_xscale('log')
axes[0].legend()

# Security vs Speed tradeoff
for l2, color in zip(l2_solutions, l2_colors):
    security = 10 if l2.security_from_l1 else 5
    axes[1].scatter(l2.tps, security, s=l2.tvl_b * 100 + 50,
                    color=color, zorder=5, alpha=0.7)
    axes[1].annotate(l2.name, (l2.tps, security), fontsize=8,
                     xytext=(5, 5), textcoords='offset points')

axes[1].set_xlabel('TPS (log scale)')
axes[1].set_ylabel('Security Level')
axes[1].set_title('L2 Security vs Speed (size = TVL)')
axes[1].set_xscale('log')

# Add legend for types
for type_name, color in type_colors.items():
    axes[1].scatter([], [], color=color, label=type_name, s=60)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/l2_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("Rollups inherit L1 security; sidechains and channels trade security for speed.")

---
## 4. Cross-Chain Bridges

Bridges transfer assets between blockchains. They are critical infrastructure but also major attack surfaces.

> **Definition: Lock-and-Mint Bridge** -- A bridge mechanism where tokens are locked on the source chain and equivalent wrapped tokens are minted on the destination chain. The bridge contract holds the locked assets.

### Bridge Security Models
- **Trusted (centralized)**: Multisig controls bridge funds (e.g., Ronin bridge)
- **Optimistic**: Assumes valid, allows challenges (e.g., Nomad)
- **Light client**: Verifies source chain proofs on destination (e.g., IBC)
- **ZK verified**: Cryptographic proof of source chain state

### Notable Bridge Hacks
- Ronin Bridge (2022): $625M -- compromised validator keys
- Wormhole (2022): $320M -- signature verification bug
- Nomad (2022): $190M -- message verification flaw

**Source:** Zhou, L. et al. (2023). "SoK: Decentralized Bridges." *IEEE S&P*.

In [ ]:
@dataclass
class BridgeProfile:
    """Cross-chain bridge profile."""
    name: str
    type: str
    tvl_m: float            # Total value locked in millions
    n_validators: int       # For multisig/validator bridges
    threshold: int          # Signature threshold (e.g., 5 of 9)
    hacked: bool
    hack_amount_m: float
    security_score: float   # 1-10


bridges = [
    BridgeProfile("Ronin", "Multisig", 600, 9, 5, True, 625, 3),
    BridgeProfile("Wormhole", "Multisig+Guardian", 2000, 19, 13, True, 320, 5),
    BridgeProfile("Nomad", "Optimistic", 0, 1, 1, True, 190, 2),
    BridgeProfile("IBC (Cosmos)", "Light Client", 5000, 0, 0, False, 0, 9),
    BridgeProfile("Polygon Bridge", "PoS Validators", 3000, 100, 67, False, 0, 7),
    BridgeProfile("Arbitrum Bridge", "L1 Verified", 4000, 0, 0, False, 0, 9),
    BridgeProfile("zkBridge", "ZK Proof", 500, 0, 0, False, 0, 10),
]

def bridge_attack_cost(bridge: BridgeProfile, validator_cost_m: float = 10) -> float:
    """Estimate cost to compromise a bridge."""
    if bridge.type in ["Light Client", "L1 Verified", "ZK Proof"]:
        return float('inf')  # Must attack the L1 itself
    if bridge.n_validators > 0:
        # Need to compromise threshold validators
        return bridge.threshold * validator_cost_m
    return 0  # Unknown


print("=" * 80)
print("CROSS-CHAIN BRIDGE SECURITY ANALYSIS")
print("=" * 80)

print(f"\n{'Bridge':<18} {'Type':<18} {'TVL':>8} {'Validators':>12} {'Hacked?':>9} {'Score':>7}")
print("-" * 75)
for b in bridges:
    hack = f'${b.hack_amount_m:.0f}M' if b.hacked else 'No'
    val_str = f"{b.threshold}/{b.n_validators}" if b.n_validators > 0 else 'N/A'
    print(f"{b.name:<18} {b.type:<18} ${b.tvl_m:>6,.0f}M {val_str:>12} {hack:>9} {b.security_score:>5}/10")

total_hacked = sum(b.hack_amount_m for b in bridges if b.hacked)
print(f"\nTotal bridge hacks: ${total_hacked:,.0f}M")
print(f"Key insight: Multisig bridges are single points of failure.")
print(f"Light client and ZK bridges inherit L1 security.")

In [ ]:
# Visualize bridge security
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bridge security vs TVL
b_names = [b.name for b in bridges]
b_scores = [b.security_score for b in bridges]
b_tvls = [b.tvl_m for b in bridges]
b_colors = ['red' if b.hacked else 'green' for b in bridges]

axes[0].scatter(b_scores, b_tvls, s=200, c=b_colors, zorder=5, alpha=0.7)
for b in bridges:
    axes[0].annotate(b.name, (b.security_score, b.tvl_m), fontsize=8,
                     xytext=(5, 5), textcoords='offset points')
axes[0].set_xlabel('Security Score (1-10)')
axes[0].set_ylabel('TVL ($M)')
axes[0].set_title('Bridge Security vs TVL (red = hacked)')

# Bridge hack timeline
hacked = [b for b in bridges if b.hacked]
hack_names = [b.name for b in hacked]
hack_amounts = [b.hack_amount_m for b in hacked]
hack_colors = ['#d62728', '#ff7f0e', '#e377c2']

bars = axes[1].barh(hack_names, hack_amounts, color=hack_colors[:len(hacked)])
axes[1].set_xlabel('Amount Lost ($M)')
axes[1].set_title('Major Bridge Hacks')
for bar, amt in zip(bars, hack_amounts):
    axes[1].text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                f'${amt:.0f}M', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('/tmp/bridge_security.png', dpi=100, bbox_inches='tight')
plt.show()
print("Bridge security analysis complete.")

---
## 5. Interoperability & Atomic Swaps

Atomic swaps allow trustless exchange of assets across different blockchains without a centralized intermediary.

> **Definition: Atomic Swap** -- A cryptographic protocol that enables the exchange of tokens between two blockchains without requiring trust. Uses Hash Time-Locked Contracts (HTLCs) to ensure either both transfers complete or neither does.

> **Definition: IBC (Inter-Blockchain Communication)** -- A protocol developed for the Cosmos ecosystem that enables sovereign blockchains to transfer tokens and data through light client verification, without bridges holding assets.

In [ ]:
class AtomicSwapSimulator:
    """Simulate Hash Time-Locked Contract (HTLC) atomic swap."""
    
    def __init__(self, chain_a: str = "Bitcoin", chain_b: str = "Ethereum") -> None:
        self.chain_a = chain_a
        self.chain_b = chain_b
        self.swap_log: List[Dict] = []
    
    def create_htlc(self, sender: str, amount: float, chain: str,
                    secret_hash: str, timeout_blocks: int) -> Dict:
        """Create an HTLC on the specified chain."""
        return {
            'sender': sender, 'amount': amount, 'chain': chain,
            'secret_hash': secret_hash, 'timeout': timeout_blocks,
            'status': 'locked'
        }
    
    def simulate_swap(self, alice_amount: float, bob_amount: float,
                      success: bool = True) -> Dict:
        """Simulate a complete atomic swap between Alice and Bob."""
        import hashlib
        
        # Step 1: Alice generates secret
        secret = "atomic_swap_secret_12345"
        secret_hash = hashlib.sha256(secret.encode()).hexdigest()
        
        steps = []
        
        # Step 2: Alice locks on Chain A
        htlc_a = self.create_htlc("Alice", alice_amount, self.chain_a,
                                   secret_hash, timeout_blocks=144)
        steps.append(f"1. Alice locks {alice_amount} on {self.chain_a} (timeout: 144 blocks)")
        
        # Step 3: Bob locks on Chain B (shorter timeout)
        htlc_b = self.create_htlc("Bob", bob_amount, self.chain_b,
                                   secret_hash, timeout_blocks=72)
        steps.append(f"2. Bob locks {bob_amount} on {self.chain_b} (timeout: 72 blocks)")
        
        if success:
            # Step 4: Alice claims on Chain B (reveals secret)
            htlc_b['status'] = 'claimed'
            steps.append(f"3. Alice claims {bob_amount} on {self.chain_b} (reveals secret)")
            
            # Step 5: Bob uses revealed secret to claim on Chain A
            htlc_a['status'] = 'claimed'
            steps.append(f"4. Bob claims {alice_amount} on {self.chain_a} (uses revealed secret)")
            steps.append("   SWAP COMPLETE: Both parties received their tokens.")
        else:
            # Timeout: both parties refunded
            htlc_a['status'] = 'refunded'
            htlc_b['status'] = 'refunded'
            steps.append("3. Timeout reached -- both HTLCs refunded.")
            steps.append("   SWAP FAILED: No tokens exchanged (atomic guarantee).")
        
        result = {
            'success': success,
            'alice_sends': (alice_amount, self.chain_a),
            'bob_sends': (bob_amount, self.chain_b),
            'steps': steps,
            'htlc_a': htlc_a,
            'htlc_b': htlc_b
        }
        self.swap_log.append(result)
        return result


# Demonstrate atomic swap
print("=" * 60)
print("ATOMIC SWAP SIMULATION (HTLC)")
print("=" * 60)

swap_sim = AtomicSwapSimulator("Bitcoin", "Ethereum")

# Successful swap
print("\n--- Successful Swap ---")
result = swap_sim.simulate_swap(alice_amount=1.0, bob_amount=30.0, success=True)
for step in result['steps']:
    print(f"  {step}")

# Failed swap (timeout)
print("\n--- Failed Swap (Timeout) ---")
result = swap_sim.simulate_swap(alice_amount=0.5, bob_amount=15.0, success=False)
for step in result['steps']:
    print(f"  {step}")

print("\nKey property: Atomicity -- either both transfers complete or neither does.")

---
## 6. Chain Metrics Dashboard

Tracking key metrics across chains reveals adoption patterns and capital flows.

In [ ]:
# Generate synthetic time-series data for chain metrics
np.random.seed(42)
months = 24  # 2 years of monthly data
time_axis = np.arange(months)

# TVL over time (synthetic)
tvl_data = {
    'Ethereum': 50 + 20 * np.sin(time_axis / 6) + np.random.normal(0, 3, months) + time_axis * 0.5,
    'Arbitrum': 0.5 + time_axis * 0.15 + np.random.normal(0, 0.3, months),
    'Solana': 2 + 3 * np.sin(time_axis / 4) + np.random.normal(0, 0.5, months) + time_axis * 0.1,
    'BNB Chain': 5 + np.random.normal(0, 0.5, months) - time_axis * 0.05,
    'Avalanche': 1.5 + np.random.normal(0, 0.3, months),
}

# Ensure non-negative
for chain in tvl_data:
    tvl_data[chain] = np.maximum(0.1, tvl_data[chain])

# Daily active addresses
daa_data = {
    'Ethereum': (400 + 100 * np.sin(time_axis / 5) + np.random.normal(0, 30, months)) * 1000,
    'Solana': (800 + time_axis * 50 + np.random.normal(0, 100, months)) * 1000,
    'BNB Chain': (600 + np.random.normal(0, 50, months)) * 1000,
    'Arbitrum': (100 + time_axis * 20 + np.random.normal(0, 20, months)) * 1000,
    'Avalanche': (50 + np.random.normal(0, 10, months)) * 1000,
}

for chain in daa_data:
    daa_data[chain] = np.maximum(10000, daa_data[chain])

# Visualize dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

chain_colors = {'Ethereum': 'blue', 'Arbitrum': '#28A0F0', 'Solana': 'purple',
                'BNB Chain': '#F3BA2F', 'Avalanche': '#E84142'}

# TVL over time
for chain, tvl in tvl_data.items():
    axes[0, 0].plot(time_axis, tvl, linewidth=2, color=chain_colors[chain], label=chain)
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('TVL ($B)')
axes[0, 0].set_title('Total Value Locked Over Time')
axes[0, 0].legend(fontsize=8)

# TVL share (stacked area)
tvl_matrix = np.array([tvl_data[c] for c in tvl_data])
tvl_pcts = tvl_matrix / tvl_matrix.sum(axis=0) * 100
axes[0, 1].stackplot(time_axis, tvl_pcts,
                     labels=list(tvl_data.keys()),
                     colors=[chain_colors[c] for c in tvl_data],
                     alpha=0.7)
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('TVL Share (%)')
axes[0, 1].set_title('TVL Market Share')
axes[0, 1].legend(fontsize=8, loc='center right')

# Daily active addresses
for chain, daa in daa_data.items():
    axes[1, 0].plot(time_axis, daa / 1000, linewidth=2, color=chain_colors[chain], label=chain)
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Daily Active Addresses (K)')
axes[1, 0].set_title('Daily Active Addresses')
axes[1, 0].legend(fontsize=8)

# Current snapshot bar chart
current_tvl = {c: tvl[-1] for c, tvl in tvl_data.items()}
sorted_chains = sorted(current_tvl, key=current_tvl.get, reverse=True)
axes[1, 1].barh(sorted_chains, [current_tvl[c] for c in sorted_chains],
                color=[chain_colors[c] for c in sorted_chains])
axes[1, 1].set_xlabel('TVL ($B)')
axes[1, 1].set_title('Current TVL Snapshot')

plt.tight_layout()
plt.savefig('/tmp/chain_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()
print("Chain metrics dashboard generated.")

---
## 7. Capital Migration Analysis

Capital flows between chains reveal user preferences and ecosystem health. Large outflows may signal declining confidence, while inflows indicate growing adoption.

In [ ]:
# Simulate capital flow matrix between chains
np.random.seed(42)

flow_chains = ['Ethereum', 'Arbitrum', 'Solana', 'BNB Chain', 'Avalanche']
n_chains = len(flow_chains)

# Monthly capital flows ($M) between chains
# Positive = flow from row to column
flow_matrix = np.zeros((n_chains, n_chains))

# Ethereum -> L2s (large outflows)
flow_matrix[0, 1] = 500   # ETH -> Arbitrum
flow_matrix[0, 2] = 200   # ETH -> Solana
flow_matrix[0, 3] = 100   # ETH -> BNB
flow_matrix[0, 4] = 50    # ETH -> Avalanche

# Return flows (smaller)
flow_matrix[1, 0] = 200   # Arbitrum -> ETH
flow_matrix[2, 0] = 150   # Solana -> ETH
flow_matrix[3, 0] = 80    # BNB -> ETH
flow_matrix[4, 0] = 30    # Avalanche -> ETH

# Cross-chain flows
flow_matrix[2, 3] = 30    # Solana -> BNB
flow_matrix[3, 2] = 20    # BNB -> Solana

# Calculate net flows
net_flows = flow_matrix.sum(axis=0) - flow_matrix.sum(axis=1)

print("=" * 60)
print("CAPITAL MIGRATION ANALYSIS")
print("=" * 60)

print(f"\nMonthly Capital Flow Matrix ($M):")
print(f"{'From / To':<12}", end='')
for c in flow_chains:
    print(f"{c:>12}", end='')
print(f"{'Net Flow':>12}")
print("-" * (12 + 12 * n_chains + 12))

for i, src in enumerate(flow_chains):
    print(f"{src:<12}", end='')
    for j in range(n_chains):
        if i == j:
            print(f"{'--':>12}", end='')
        else:
            print(f"${flow_matrix[i,j]:>10,.0f}M", end='')
    color = '+' if net_flows[i] > 0 else ''
    print(f"${color}{net_flows[i]:>9,.0f}M")

print(f"\nBiggest net inflow: {flow_chains[np.argmax(net_flows)]} (+${max(net_flows):,.0f}M)")
print(f"Biggest net outflow: {flow_chains[np.argmin(net_flows)]} (${min(net_flows):,.0f}M)")

In [ ]:
# Visualize capital flows
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Net flows bar chart
bar_colors = ['green' if n > 0 else 'red' for n in net_flows]
axes[0].barh(flow_chains, net_flows, color=bar_colors)
axes[0].axvline(x=0, color='black', linewidth=0.5)
axes[0].set_xlabel('Net Capital Flow ($M/month)')
axes[0].set_title('Net Capital Flows Between Chains')

for i, (flow, name) in enumerate(zip(net_flows, flow_chains)):
    sign = '+' if flow > 0 else ''
    axes[0].text(flow + (20 if flow > 0 else -20),
                i, f'{sign}${flow:,.0f}M',
                va='center', ha='left' if flow > 0 else 'right', fontsize=9)

# Dominance index over time
total_tvl_over_time = sum(tvl_data[c] for c in tvl_data)
eth_dominance = tvl_data['Ethereum'] / total_tvl_over_time * 100

axes[1].plot(time_axis, eth_dominance, 'b-', linewidth=2, label='Ethereum')
axes[1].fill_between(time_axis, eth_dominance, alpha=0.1, color='blue')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('TVL Dominance (%)')
axes[1].set_title('Ethereum TVL Dominance Over Time')
axes[1].legend()

plt.tight_layout()
plt.savefig('/tmp/capital_flows.png', dpi=100, bbox_inches='tight')
plt.show()
print("Capital is flowing from Ethereum mainnet to L2s and alt-L1s.")

---
## Exercises

### Exercise 1: L2 Fee Savings Calculator

Build a calculator that estimates how much a user saves by using different L2s compared to Ethereum L1 for common transaction types.

**Hints:**
- Common transaction types: ETH transfer, ERC-20 transfer, swap, NFT mint
- Each has different gas costs on L1
- L2s have different compression ratios and base fees
- Factor in L1 data posting costs for rollups

In [ ]:
class L2FeeSavings:
    """Calculate fee savings from using L2s."""
    
    def __init__(self, l1_gas_price_gwei: float = 30,
                 eth_price: float = 2000) -> None:
        """Initialize with current L1 conditions."""
        self.l1_gas_price = l1_gas_price_gwei
        self.eth_price = eth_price
        # YOUR CODE HERE
    
    def estimate_l1_cost(self, tx_type: str) -> float:
        """Estimate L1 gas cost for a transaction type."""
        # YOUR CODE HERE
        pass
    
    def estimate_l2_cost(self, tx_type: str, l2_name: str) -> float:
        """Estimate L2 cost for same transaction."""
        # YOUR CODE HERE
        pass
    
    def compare_all(self, tx_type: str) -> Dict[str, float]:
        """Compare costs across all L2s for a transaction type."""
        # YOUR CODE HERE
        pass

### Exercise 2: Bridge Risk Scorer

Build a risk scoring model for cross-chain bridges based on their architecture, validator count, audit history, and TVL.

**Hints:**
- Multisig bridges score lower than light client bridges
- More validators = higher security (but diminishing returns)
- Audit count and bug bounty size matter
- Higher TVL = bigger target = need more security

In [ ]:
class BridgeRiskScorer:
    """Risk scoring model for cross-chain bridges."""
    
    def __init__(self) -> None:
        """Initialize scorer with default weights."""
        # YOUR CODE HERE
        pass
    
    def score_bridge(self, bridge: BridgeProfile,
                     audits: int = 0, bug_bounty_m: float = 0) -> Dict:
        """Score a bridge's risk profile."""
        # YOUR CODE HERE
        pass
    
    def recommend_bridge(self, amount: float,
                         bridges: List[BridgeProfile]) -> str:
        """Recommend the safest bridge for a given transfer amount."""
        # YOUR CODE HERE
        pass

### Exercise 3: Chain Migration Predictor

Build a model that predicts capital flows between chains based on fee differences, TPS ratios, and developer activity.

**Hints:**
- Users migrate to lower-fee chains (price elasticity)
- Developer activity predicts future TVL growth
- Network effects create inertia (users prefer chains where others already are)
- Use a gravity model: flow proportional to TVL product, inversely proportional to friction

In [ ]:
class ChainMigrationPredictor:
    """Predict capital flows between chains."""
    
    def __init__(self, chains: List[ChainProfile]) -> None:
        """Initialize with chain profiles."""
        self.chains = chains
        # YOUR CODE HERE
    
    def gravity_model(self, chain_a: str, chain_b: str) -> float:
        """Predict flow using gravity model."""
        # YOUR CODE HERE
        pass
    
    def predict_flows(self, months_ahead: int = 6) -> Dict:
        """Predict capital flows for next N months."""
        # YOUR CODE HERE
        pass

---
## Summary

### What You Learned
- [x] Blockchain trilemma quantification across security, scalability, and decentralization
- [x] L1 blockchain comparison: TPS, finality, fees, validator counts
- [x] L2 scaling solutions: optimistic rollups, ZK rollups, state channels, sidechains
- [x] Cross-chain bridge architectures and their security tradeoffs
- [x] Atomic swap mechanics using Hash Time-Locked Contracts (HTLCs)
- [x] Chain metrics dashboard construction and analysis
- [x] Capital migration patterns between blockchain ecosystems

### Key Takeaways
1. **The trilemma is real** -- no chain excels at all three dimensions simultaneously
2. **L2s inherit L1 security** (rollups) or sacrifice it (sidechains) for scalability
3. **Bridges are the weakest link** -- billions have been lost to bridge exploits
4. **Light client and ZK bridges are most secure** -- they don't rely on trusted intermediaries
5. **Capital is fragmenting** -- TVL is spreading across more chains, reducing Ethereum dominance
6. **Interoperability is the next frontier** -- seamless cross-chain UX will drive adoption

### Further Reading
- Buterin, V. (2021). "Why sharding is great." vitalik.eth.limo
- L2BEAT. (2024). "Layer 2 Risk Analysis." l2beat.com
- Zhou, L. et al. (2023). "SoK: Decentralized Bridges." IEEE S&P.

### Next Steps
- [Notebook 16: Energy & Sustainability](16-energy-sustainability.ipynb) -- Environmental impact of blockchains
- [Section 05: Platform Comparison](../sections/05-platform-comparison.md) -- Architecture deep dive